In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
dbutils.widgets.text("env", "dev", "Ambiente (dev | prod)")
dbutils.widgets.text("container", "raw")
dbutils.widgets.text("catalog_base", "catalog_pr_db")
dbutils.widgets.text("esquema", "bronze")
dbutils.widgets.text("storageName", "adlsdvbtdev01")


env = dbutils.widgets.get("env").strip().lower()
container = dbutils.widgets.get("container")
catalogo = f"{dbutils.widgets.get('catalog_base')}_{env}"
esquema = dbutils.widgets.get("esquema")
storageName = dbutils.widgets.get("storageName")
 
assert env in ("dev", "prod"), f"env inválido: {env}"
 
ruta = f"abfss://{container}@{storageName}.dfs.core.windows.net/races.csv"
print(f"env={env} | catalogo={catalogo} | ruta={ruta}")


In [0]:
df_races= spark.read.option('header', True)\
                        .option('inferSchema', True)\
                        .csv(ruta)

In [0]:
df_races.display()

In [0]:
races_schema = StructType(fields=[StructField("raceId", IntegerType(), False),
                                  StructField("year", IntegerType(), True),
                                  StructField("round", IntegerType(), True),
                                  StructField("circuitId", IntegerType(), True),
                                  StructField("name", StringType(), True),
                                  StructField("date", DateType(), True),
                                  StructField("time", StringType(), True),
                                  StructField("url", StringType(), True) 
                                  ])

In [0]:
races_schema

In [0]:
races_df = spark.read \
            .option("header", True) \
            .schema(races_schema) \
            .csv(ruta)

In [0]:
races_with_timestamp_df = races_df.withColumn("ingestion_date", current_timestamp())

In [0]:
races_selected_df = races_with_timestamp_df.select(col('raceId').alias('race_id'), 
                                                   col('year').alias('race_year'), 
                                                   col('round'), col('circuitId').alias('circuit_id'),
                                                   col('date').alias('date_race'), 
                                                   col('name'), col('ingestion_date'))

In [0]:
print(f"catalogo = {catalogo}")
print(f"esquema = {esquema}")

In [0]:
spark.sql(f"USE CATALOG `{catalogo}`")
spark.sql(f"USE SCHEMA `{esquema}`")

In [0]:
races_selected_df.display()

In [0]:
print(f"current_catalog = {spark.sql('SELECT current_catalog()').collect()[0][0]}")
print(f"current_schema = {spark.sql('SELECT current_schema()').collect()[0][0]}")


In [0]:
races_selected_df.write.mode("overwrite").partitionBy('race_year').saveAsTable(f"{catalogo}.{esquema}.races")

In [0]:
%sql
--select * from catalog_pr_db_dev.bronze.races